# Basic classification + noise — L1 lab

Multi-label text classification on three datasets, two classifiers (TF-IDF+LogReg and LLM few-shot), three demos that all land on the same L1 punchline: **a single accuracy number is one draw**.

Source of uncertainty in focus: **sampling noise**.

## Setup

In [ ]:
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
load_dotenv()

from lab import (
    load_bitext, load_goemotions, load_synthetic_itsm,
    label_set_of,
    TfidfLogRegClassifier, LlmFewShotClassifier,
    evaluate, run_one,
)

pd.set_option('display.max_colwidth', 80)

## Datasets

Three datasets loaded from `../../datasets/`. The first time you run this, Bitext and GoEmotions pull from HuggingFace and cache a small subset locally. The synthetic ITSM set is generated by `datasets/synthetic_itsm/generate.py`.

In [ ]:
bitext     = load_bitext(n=2000)
goemotions = load_goemotions(n=2000)
synthetic  = load_synthetic_itsm()

for name, df in [('bitext', bitext), ('goemotions', goemotions), ('synthetic', synthetic)]:
    if df is None:
        print(f'{name}: not generated yet — run datasets/synthetic_itsm/generate.py')
    else:
        n_labels = len(label_set_of(df))
        print(f'{name}: {len(df)} rows, {n_labels} distinct labels')
        print(df.head(3))
        print()

## Demo A — Per-label F1 on a subsample

Pick a dataset, sample 300 cases, train TF-IDF on 70%, run both classifiers on the rest, look at per-label F1.

The headline numbers (`micro_f1`, `macro_f1`) hide the rare-label story.

In [ ]:
result = run_one(goemotions, n_sample=300, seed=0)
print('TF-IDF + LogReg:')
print(f"  micro_f1 = {result['tfidf']['micro_f1']:.3f}")
print(f"  macro_f1 = {result['tfidf']['macro_f1']:.3f}")
print('LLM few-shot (mock if LIVE!=true):')
print(f"  micro_f1 = {result['llm']['micro_f1']:.3f}")
print(f"  macro_f1 = {result['llm']['macro_f1']:.3f}")

# Per-label F1, side by side, sorted by TF-IDF score
per_label = pd.DataFrame({
    'tfidf': result['tfidf']['per_label'],
    'llm':   result['llm']['per_label'],
}).sort_values('tfidf', ascending=False)
print()
print(per_label)

## Demo B — Resampling spread

Run Demo A 20 times with different random subsamples. The headline F1 wobbles; **rare labels wobble a lot**.

This is the L1 sampling-noise punchline: a single F1 is one draw.

In [ ]:
n_repeats = 20
n_sample  = 300
rows = []
for seed in range(n_repeats):
    r = run_one(goemotions, n_sample=n_sample, seed=seed)
    rows.append({
        'seed': seed,
        'tfidf_micro': r['tfidf']['micro_f1'],
        'tfidf_macro': r['tfidf']['macro_f1'],
        'llm_micro':   r['llm']['micro_f1'],
        'llm_macro':   r['llm']['macro_f1'],
    })
df_b = pd.DataFrame(rows)
print(df_b.describe().loc[['mean','std','min','max']].round(3))

fig, ax = plt.subplots(figsize=(8, 4))
df_b[['tfidf_micro','tfidf_macro','llm_micro','llm_macro']].plot(kind='box', ax=ax)
ax.set_title(f'F1 spread across {n_repeats} resamples of {n_sample} cases (GoEmotions)')
ax.set_ylabel('F1')
plt.tight_layout()
plt.show()

In [ ]:
# Per-label spread — rare labels wobble a lot more than common ones
per_label_runs = []
for seed in range(n_repeats):
    r = run_one(goemotions, n_sample=n_sample, seed=seed)
    per_label_runs.append(r['tfidf']['per_label'])
pl_df = pd.DataFrame(per_label_runs).fillna(0)
spread = pl_df.agg(['mean','std']).T.sort_values('mean', ascending=False)
print(spread.round(3))

fig, ax = plt.subplots(figsize=(10, 4))
spread['mean'].plot(kind='bar', yerr=spread['std'], ax=ax)
ax.set_title('Per-label F1 (mean ± std across resamples) — TF-IDF on GoEmotions')
ax.set_ylabel('F1')
plt.xticks(rotation=70)
plt.tight_layout()
plt.show()

## Demo C — Skew slider on the synthetic ITSM dataset

Generate the synthetic dataset at different skew levels and compare:

- the TF-IDF+LogReg classifier
- a trivial "predict the K most common labels" baseline

As skew grows, the trivial baseline catches up on `micro_f1` (which is dominated by the common labels), but it never makes up the gap on `macro_f1` (which gives every label equal weight). The headline number flatters the trivial baseline; the macro number exposes it.

In [ ]:
import subprocess
from collections import Counter

GEN = '../../datasets/synthetic_itsm/generate.py'
DATA = '../../datasets/synthetic_itsm/data.csv'

def regen(skew, n=1000):
    subprocess.run(['python3', GEN, '--n', str(n), '--skew', skew], check=True, capture_output=True)
    return load_synthetic_itsm()

class MajorityBaseline:
    def __init__(self, k=3):
        self.k = k
        self.top = []
    def fit(self, _texts, label_lists):
        c = Counter(l for labs in label_lists for l in labs)
        self.top = [l for l,_ in c.most_common(self.k)]
        return self
    def predict(self, texts):
        return [list(self.top) for _ in texts]

def run_with_baseline(df, n_sample=400, seed=0):
    sub = df.sample(n=min(n_sample, len(df)), random_state=seed).reset_index(drop=True)
    n_train = int(0.7 * len(sub))
    train, test = sub.iloc[:n_train], sub.iloc[n_train:]
    labels = label_set_of(df)
    tfidf = TfidfLogRegClassifier().fit(train['text'].tolist(), train['labels'].tolist())
    base  = MajorityBaseline(k=3).fit(train['text'].tolist(),   train['labels'].tolist())
    y_true = test['labels'].tolist()
    return {
        'tfidf':    evaluate(y_true, tfidf.predict(test['text'].tolist()), labels),
        'majority': evaluate(y_true, base.predict( test['text'].tolist()), labels),
    }

rows = []
for skew in ['flat','default','heavy']:
    df_s = regen(skew, n=1000)
    r = run_with_baseline(df_s, n_sample=400, seed=0)
    rows.append({
        'skew': skew,
        'tfidf_micro':    r['tfidf']['micro_f1'],
        'tfidf_macro':    r['tfidf']['macro_f1'],
        'majority_micro': r['majority']['micro_f1'],
        'majority_macro': r['majority']['macro_f1'],
    })
df_c = pd.DataFrame(rows)
print(df_c.round(3))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df_c.set_index('skew')[['tfidf_micro','majority_micro']].plot(kind='bar', ax=axes[0])
axes[0].set_title('micro_f1 — common-label-dominated')
axes[0].set_ylim(0, 1)
df_c.set_index('skew')[['tfidf_macro','majority_macro']].plot(kind='bar', ax=axes[1])
axes[1].set_title('macro_f1 — every label equal')
axes[1].set_ylim(0, 1)
plt.tight_layout()
plt.show()

## The L1 takeaway

Three things you should have *felt* in this lab:

1. **One headline number hides a per-label story.** Demo A's micro/macro F1 are summaries — the per-label table shows where the model actually fails.
2. **A single F1 is one draw.** Demo B's box plot shows how much your headline number can move just by reshuffling the same 300 examples. Rare-label F1 wobbles much more than common-label F1.
3. **Aggregate metrics can flatter trivial models.** Demo C shows a "predict-the-common-tags" baseline catching up to a real classifier on `micro_f1` as the dataset skews — and the gap on `macro_f1` is the only honest signal.

All three are L1's sampling-noise dimension in disguise. The course returns to these ideas: Demo A → L3 (multi-label metrics), Demo B → L4 (sampling noise → multiple-comparisons), Demo C → L5 (overfitting & honest comparison).